# Astra - Clinical Diagnostic Triage Pipeline
### Two-Stage RAG Architecture for Symptom Analysis & Follow-up Generation

1. **Stage 1: Vector Retrieval (Pinecone)**
   - Converts patient symptoms to 1536-dimensional embeddings with OpenAI `text-embedding-3-small`.
   - Queries Pinecone index `astra-diseases` to retrieve candidate diseases and their known symptom profiles.

2. **Stage 2: LLM Reasoning & Follow-up Generation (OpenAI GPT-4o-mini)**
   - Analyzes patient symptoms against retrieved candidate diseases.
   - Detects differentiating/discriminating symptoms to distinguish between conditions.
   - Generates targeted, clinical follow-up questions (with optional multilingual support) and screens for emergency red flags.

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

# 1. Load Environment Variables from .env
load_dotenv('.env')
OPENAI_API_KEY = os.getenv('OPEN_AI_API')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX', 'astra-diseases')

# 2. Initialize Clients
openai_client = OpenAI(api_key=OPENAI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

print("Pinecone Connected!")
print("Index Stats:", index.describe_index_stats())

C:\Users\sunil\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pinecone Connected!
Index Stats: {'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 773}},
 'total_vector_count': 773,
 'vector_type': 'dense'}


## 1. Stage 1: Vector Search in Pinecone (`astra-diseases`)

In [2]:
def retrieve_candidate_diseases(symptoms_text: str, top_k: int = 5):
    """
    Embeds the input symptoms and retrieves top_k candidate diseases from Pinecone.
    """
    # Embed using text-embedding-3-small (1536 dim)
    res = openai_client.embeddings.create(
        input=[symptoms_text.strip()],
        model="text-embedding-3-small"
    )
    query_vector = res.data[0].embedding

    # Query Pinecone
    query_results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    candidates = []
    for match in query_results.matches:
        metadata = match.metadata or {}
        candidates.append({
            "id": match.id,
            "disease": metadata.get("disease", ""),
            "similarity_score": round(match.score, 4),
            "characteristic_symptoms": metadata.get("symptoms", "")
        })
    return candidates

# Quick Test
sample_complaint = "fever, shortness of breath, sharp chest pain, cough"
candidates = retrieve_candidate_diseases(sample_complaint, top_k=4)
for c in candidates:
    print(f"[{c['similarity_score']:.4f}] {c['disease']}: {c['characteristic_symptoms']}")

[0.7638] pulmonary congestion: shortness of breath, sharp chest pain, chest tightness, cough, nasal congestion, leg swelling, weakness, decreased heart rate, fever
[0.7472] pneumonia: shortness of breath, sharp chest pain, sore throat, cough, nasal congestion, vomiting, wheezing, weakness, fever, difficulty breathing, chills, coryza
[0.7461] pulmonary fibrosis: shortness of breath, sharp chest pain, chest tightness, cough, fainting, fever, difficulty breathing
[0.7242] atelectasis: shortness of breath, sharp chest pain, dizziness, sore throat, cough, headache


## 2. Stage 2: OpenAI LLM Layer for Targeted Follow-up Questions

In [3]:
def generate_followup_questions(patient_symptoms: str, language: str = "English", top_k: int = 4, model: str = "gpt-4o-mini"):
    """
    Fetches candidate diseases from Pinecone and calls OpenAI LLM to formulate follow-up questions.
    """
    # 1. Fetch Candidates from Pinecone
    candidates = retrieve_candidate_diseases(patient_symptoms, top_k=top_k)

    # 2. System and User Prompt
    system_prompt = (
        "You are an expert Clinical Diagnostic & Triage AI at a healthcare clinic. "
        "Analyze the patient's reported symptoms alongside disease candidates retrieved from Pinecone. "
        "Generate targeted, high-yield follow-up questions to differentiate between suspected conditions "
        "and identify any critical red flags. Never repeat symptoms already disclosed. "
        "Respond ONLY with a strictly valid JSON object."
    )

    user_prompt = f"""
Patient Stated Complaint: "{patient_symptoms}"

Suspected Candidate Diseases (Retrieved from Pinecone vector DB):
{json.dumps(candidates, indent=2)}

Language Preference: {language}

CLINICAL INSTRUCTIONS:
1. Compare reported symptoms against candidate diseases.
2. Determine missing discriminating symptoms that distinguish between the top candidate diseases.
3. Formulate targeted follow-up questions.
4. If Language is not English, output the question in {language} and provide English translation.
5. Suspected diseases are only predicted data so only take those data which is reletivant with Patient Stated Complaint and help to make usefuk follow up question.
JSON Schema:
{{
  "followup_questions": [
    {{
      "question_id": 1,
      "question_text": "<question phrased in {language}>",
      "question_text_en": "<question in English>",
      "clinical_intent": "<intent>",
      "target_differential": "<disease or symptom targeted>"
    }}
  ],
}}
"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.2
    )

    result = json.loads(response.choices[0].message.content)
    result["pinecone_matches"] = candidates
    return result

## 3. Example 1: Respiratory Complaint (Pneumonia vs Pulmonary Fibrosis/Congestion)

In [4]:
patient_input_1 = " Fever for two days with associated sore throat, body aches, headache, stomach ache, one episode of vomiting, and occasional dizziness."
result_1 = generate_followup_questions(patient_input_1, language="English")





print("\n=== GENERATED FOLLOW-UP QUESTIONS ===")
for q in result_1.get("followup_questions", []):
    print(f"Q{q['question_id']}: {q['question_text']}")
    print(f"   Intent: {q['clinical_intent']} (Target: {q['target_differential']})")




=== GENERATED FOLLOW-UP QUESTIONS ===
Q1: Have you experienced any neck pain or stiffness?
   Intent: To assess for signs of meningitis (Target: meningitis)
Q2: Are you having any difficulty swallowing or a rash?
   Intent: To evaluate for strep throat (Target: strep throat)
Q3: Have you had any nasal congestion or cough?
   Intent: To differentiate between flu and common cold (Target: common cold)


## 4. Example 2: Abdominal Complaint with Multilingual Support (Hindi)

In [13]:
patient_input_2 = "Severe sharp abdominal pain, nausea, and vomiting"
result_2 = generate_followup_questions(patient_input_2, language="Hindi")

print("=== GENERATED FOLLOW-UP QUESTIONS (HINDI) ===")
for q in result_2.get("followup_questions", []):
    print(f"Q{q['question_id']}: {q['question_text']}")
    print(f"   English: {q.get('question_text_en')}")
    print(f"   Intent : {q['clinical_intent']}\n")

=== GENERATED FOLLOW-UP QUESTIONS (HINDI) ===
Q1: क्या आपको दस्त हो रहे हैं?
   English: Are you experiencing diarrhea?
   Intent : To assess for intestinal disease

Q2: क्या आपको खून vomit करने का अनुभव हुआ है?
   English: Have you experienced vomiting blood?
   Intent : To assess for gastritis

Q3: क्या आपको पेट के ऊपरी हिस्से में दर्द हो रहा है?
   English: Are you experiencing pain in the upper abdomen?
   Intent : To assess for gastroparesis

Q4: क्या आपके पैरों या हाथों में सूजन है?
   English: Do you have swelling in your legs or arms?
   Intent : To assess for celiac disease



## 5. Interactive Testing: Try Your Own Symptoms

In [14]:
# Change this symptom description to test any case:
my_symptoms = "Sudden dizziness, severe headache, and tingling sensation in left arm"
my_language = "Odia"  # "English", "Hindi", "Odia"

my_result = generate_followup_questions(my_symptoms, language=my_language)

print(f"Input: '{my_symptoms}'\n")
print("Suspected Diseases from Pinecone:")
for c in my_result.get("retrieved_candidates", []):
    print(f"- {c['disease']} ({c['similarity_score']:.4f})")

print("\nFollow-up Questions:")
for q in my_result.get("followup_questions", []):
    print(f"Q{q['question_id']}: {q['question_text']}")

Input: 'Sudden dizziness, severe headache, and tingling sensation in left arm'

Suspected Diseases from Pinecone:
- cerebral edema (0.5951)
- transient ischemic attack (0.5291)
- vertebrobasilar insufficiency (0.5218)
- orthostatic hypotension (0.5155)

Follow-up Questions:
Q1: ଆପଣଙ୍କର ମୁଣ୍ଡ ଘୁରାଇବା ସମୟରେ କିଛି ଅନ୍ୟାନ୍ୟ ଲକ୍ଷଣ ଅନୁଭବ କରୁଛନ୍ତି କି? (ଯଥା: ବାକ୍ରତା, କଥା କହିବାରେ କଷ୍ଟ, ଅସାଧାରଣ ଅନୁଭବ)
Q2: ଆପଣଙ୍କର ମୁଣ୍ଡ ଘୁରାଇବା ସମୟରେ ମୁଣ୍ଡରେ ବା ଦେହରେ କିଛି ଦୁଃଖ ଅନୁଭବ କରୁଛନ୍ତି କି?
Q3: ଆପଣଙ୍କର ହୃଦୟ ଧଡ଼କା କିମ୍ବା ଅସାଧାରଣ ଅନୁଭବ କରୁଛନ୍ତି କି?
Q4: ଆପଣଙ୍କୁ ମୁଣ୍ଡ ଘୁରାଇବା ସମୟରେ ବାହୁରେ କିଛି ଦୁଃଖ ଅନୁଭବ କରୁଛନ୍ତି କି?
